# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [1]:
!cat publications.tsv

'cat' is not recognized as an internal or external command,
operable program or batch file.


## Import pandas

We are using the very handy pandas library for dataframes.

In [2]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [3]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url,slides_url
0,2024-01-01,Automatic Ulos Coloring Using Genetic Algorith...,Ssrn,NaN,"Barus, A. C., Situmeang, S., Napitupulu, M. A....",automatic-ulos-coloring-using-genetic-algorith...,NaN,NaN
1,2024-01-01,Generating New Ulos Motif with Generative AI M...,International Journal of Advanced Computer Sci...,NaN,"Simanjuntak, H., Panjaitan, E., Siregar, S., M...",generating-new-ulos-motif-with-generative-ai-m...,NaN,NaN
2,2023-01-01,A Deep Learning-Based Regression Approach to I...,ACM International Conference Proceeding Series,NaN,"Situmeang, S. I. G., Sihite, R. M. G. T., Sima...",a-deep-learning-based-regression-approach-to-i...,NaN,NaN
3,2023-01-01,A cluster and association analysis visualizati...,International Journal of Informatics and Commu...,NaN,"Tamba, A. R., Lumbantoruan, K., Pakpahan, A., ...",a-cluster-and-association-analysis-visualizati...,NaN,NaN
4,2023-01-01,Predicting Students Performance Using Data Min...,Proceedings of 2023 IEEE International Confere...,NaN,"Bonar Sirait, J. C., Togatorop, P. R., Tambuna...",predicting-students-performance-using-data-min...,NaN,NaN
5,2020-01-01,Android Malware Classification Based on Permis...,Proceedings of the 5th International Conferenc...,Mobile malware has become the centerpiece of m...,"Turnip, Togu Novriansyah, Situmorang, Amsal, L...",android-malware-classification-based-on-permis...,https://doi.org/10.1145/3427423.3427427,NaN
6,2019-01-01,Movie Summarization based on Indonesian Subtit...,2019 International Conference on Sustainable I...,NaN,"Situmeang, Samuel I. G., Lubis, Ramosan K., Si...",movie-summarization-based-on-indonesian-subtit...,https://doi.org/10.1109/SIET48054.2019.8986127,NaN
7,2017-01-01,Badminton video analysis based on spatiotempor...,ICMR 2017 - Proceedings of the 2017 ACM Intern...,NaN,"Chu, W.-T. & Situmeang, S. (2017). ""Badminton ...",badminton-video-analysis-based-on-spatiotempor...,NaN,NaN
8,2016-01-01,Noise reduction in breath sound files using wa...,International Conference on Electrical Enginee...,NaN,"Syahputra, M. F., Situmeang, S. I. G., Rahmat,...",noise-reduction-in-breath-sound-files-using-wa...,NaN,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [4]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [5]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.slides_url)) > 5:
        md += "\nslidesurl: '" + item.slides_url + "'"

    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"

    if len(str(item.slides_url)) > 5:
        md += "\n[Download slides here](" + item.slides_url + ")\n" 

    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [6]:
!ls ../_publications/

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

'cat' is not recognized as an internal or external command,
operable program or batch file.
